[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/powerzoojax/PowerZooPy/blob/main/docs/en/examples/notebooks/nb08_data_advanced.ipynb)

# NB08 — Time-Series & Custom Tasks

> **Prerequisites**: [NB07 — Evaluation](./nb07_evaluation.ipynb)  
> **Time**: ~20 minutes

> **Note**: This notebook covers advanced topics. The `DataLoader` API may evolve
> as the data layer stabilises. Check `powerzoo.__version__` if you encounter issues.

## What You'll Learn

1. Load real grid time-series data with `DataLoader` and explore it visually
2. Understand how data splits in tasks map to the underlying time series
3. Write a custom reward function using `RewardFunction`
4. Register a new task variant with `register_task()`

## Setup

In [ ]:
import powerzoo
print(f"PowerZoo version: {powerzoo.__version__}")

In [ ]:
from powerzoo.data.data_loader import DataLoader
from powerzoo.tasks import make_task_env, register_task, list_tasks
from powerzoo.tasks.base import MultiAgentTask
from powerzoo.wrappers import GymnasiumWrapper
from powerzoo.benchmarks.policies import RandomPolicy
from powerzoo.benchmarks import evaluate
from powerzoo.tasks.rewards import RewardFunction

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## 1. Available Datasets

In [ ]:
loader = DataLoader()

datasets = loader.list_available_datasets()
print("Available datasets:")
for ds in datasets:
    loader.print_dataset_info(ds)

## 2. Load and Explore a Bundled Demand Dataset

In [ ]:
# Load one year of system demand (bundled example dataset)
df_demand = loader.load_data(
    dataset_name='GB_NESO_Demand_2009_2025_30min',
    start_date='2024-01-01',
    end_date='2024-12-31',
)

print(f"Shape: {df_demand.shape}")
print(f"Columns: {list(df_demand.columns)}")
print(f"\nDate range: {df_demand['datetime'].min()} → {df_demand['datetime'].max()}")
df_demand.head(3)

In [ ]:
# Plot weekly demand pattern
df_week = df_demand.head(48 * 7)  # first week
demand_col = [c for c in df_demand.columns if c != 'datetime'][0]

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(df_week['datetime'], df_week[demand_col], color='steelblue', linewidth=1.2)
ax.set_xlabel('Date')
ax.set_ylabel('Demand (MW)')
ax.set_title('System demand — first week of 2024 (30-min resolution)')

# Mark weekends
for dt in df_week['datetime']:
    if dt.dayofweek >= 5:  # weekend
        ax.axvspan(dt, dt + pd.Timedelta(minutes=30), alpha=0.03, color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# Load generation mix dataset — auto-detect by column name
df_gen = loader.load_data(
    columns=['Wind Offshore', 'Wind Onshore', 'Solar', 'Nuclear', 'Fossil Gas'],
    start_date='2024-06-01',
    end_date='2024-06-07',
)

print(f"Loaded {len(df_gen)} rows with columns: {list(df_gen.columns)}")

# Area chart: generation mix
gen_cols = ['Wind Offshore', 'Wind Onshore', 'Solar', 'Nuclear', 'Fossil Gas']
available = [c for c in gen_cols if c in df_gen.columns]

fig, ax = plt.subplots(figsize=(13, 4))
ax.stackplot(
    df_gen['datetime'],
    [df_gen[c].fillna(0) for c in available],
    labels=available,
    colors=['#1565C0', '#42A5F5', '#FDD835', '#9E9E9E', '#FF7043'],
    alpha=0.85,
)
ax.set_xlabel('Date')
ax.set_ylabel('Generation (MW)')
ax.set_title('Generation mix — June 1–7, 2024')
ax.legend(loc='upper right', fontsize=8, ncol=3)
plt.tight_layout()
plt.show()

## 3. Data Splits and the Tasks

The task `SPLIT_DATES` directly reference ranges in the DataLoader datasets.
Here's how they align:

In [ ]:
from powerzoo.tasks.simple.task1_marl_opf import MARLOPFTask

print("Task SPLIT_DATES → DataLoader date range mapping")
print("-" * 60)

for split, (start, end) in MARLOPFTask.SPLIT_DATES.items():
    df = loader.load_data(
        dataset_name='GB_NESO_Demand_2009_2025_30min',
        start_date=start,
        end_date=end,
    )
    n_steps = len(df)
    n_episodes = n_steps // 48  # 48 steps per episode
    print(f"  {split:6s}: {start} → {end}  |  {n_steps:6d} rows  ~  {n_episodes} episodes")

print()
print("Note: each episode samples a contiguous 48-step (1-day) window from the split.")

## 4. Writing a Custom Reward Function

PowerZoo's modular reward system lets you subclass `RewardFunction`
to define any objective. This reward penalizes renewable curtailment
(wasted wind/solar generation).

In [ ]:
class RenewableFriendlyReward(RewardFunction):
    """Reward that incentivizes absorbing renewable generation.

    Extends the standard economic dispatch reward with a bonus for
    reducing renewable curtailment (unused wind/solar capacity).

    Args:
        cost_weight:        Weight on generation cost term.
        safety_weight:      Weight on constraint violation penalty.
        renewable_bonus:    Bonus per MWh of renewable energy absorbed.
    """

    def __init__(
        self,
        cost_weight: float = 1.0,
        safety_weight: float = 0.5,
        renewable_bonus: float = 0.1,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.cost_weight = cost_weight
        self.safety_weight = safety_weight
        self.renewable_bonus = renewable_bonus

    def compute(self, state: dict, info: dict) -> float:
        reward = 0.0

        # Standard cost term: penalize expensive generation
        generation_cost = info.get('generation_cost', 0.0)
        reward -= self.cost_weight * generation_cost

        # Safety term: penalize constraint violations
        if not state.get('is_safe', True):
            n_violations = len(
                state.get('safety_info', {}).get('unsafe_line_ids', [])
            )
            reward -= self.safety_weight * 100.0 * n_violations

        # Renewable bonus: reward absorption of renewable generation
        renewable_absorbed = info.get('renewable_absorbed_mwh', 0.0)
        reward += self.renewable_bonus * renewable_absorbed

        return reward


# Test instantiation
rf = RenewableFriendlyReward(renewable_bonus=0.2)
test_reward = rf.compute(
    state={'is_safe': True},
    info={'generation_cost': 500.0, 'renewable_absorbed_mwh': 100.0}
)
print(f"Custom reward (cost=500, 100 MWh renewable absorbed): {test_reward:.1f}")
print(f"  = -1.0×500 + 0.2×100 = {-500 + 0.2*100:.1f}  ✓")

## 5. Registering a Custom Task

Use `register_task()` to add a task variant to the global registry.
It then becomes available via `make_task_env('my_task')`.

In [ ]:
from powerzoo.tasks.simple.task1_marl_opf import MARLOPFTask


class MARLOPFLongTask(MARLOPFTask):
    """OPF task with 7-day episodes for long-horizon RL research."""

    name = "marl_opf_long"  # unique identifier for the registry
    description = "Multi-Agent OPF — 7-day episode (long-horizon variant)"

    def __init__(self, **kwargs):
        # Default to 1-week episodes (7 days × 48 steps/day)
        kwargs.setdefault('max_steps', 336)
        super().__init__(**kwargs)


# Register — force=True allows re-running this cell
register_task('marl_opf_long', MARLOPFLongTask, force=True)

print("Registered tasks (showing OPF variants):")
for t in list_tasks():
    if 'opf' in t:
        print(f"  {t}")

# Test the new task
env = make_task_env('marl_opf_long', split='train')
gym_env = GymnasiumWrapper(env)
obs, _ = gym_env.reset(seed=42)
print(f"\nmarl_opf_long: obs_dim={obs.shape[0]}, act_dim={gym_env.action_space.shape[0]}")

In [ ]:
# Quick sanity check: run one episode to verify it works
policy = RandomPolicy(gym_env.action_space, seed=0)
result = evaluate(policy, gym_env, n_episodes=1, verbose=True)
print(f"\nEpisode length: {result['mean_ep_length']:.0f} steps (expected 336)")

## 6. Resampling Data for Different Time Resolutions

The `DataLoader` supports upsampling and downsampling to different frequencies.
This is useful if you want to prototype with a coarser resolution.

In [ ]:
# Original 30-min data
df_30min = loader.load_data(
    dataset_name='GB_NESO_Demand_2009_2025_30min',
    start_date='2024-01-01',
    end_date='2024-01-07',
)

# Resample to 60-min (downsampling: average over 2 steps)
df_60min = loader.load_data(
    dataset_name='GB_NESO_Demand_2009_2025_30min',
    start_date='2024-01-01',
    end_date='2024-01-07',
    resample='60min',
)

demand_col = [c for c in df_30min.columns if c != 'datetime'][0]

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(df_30min['datetime'], df_30min[demand_col], color='steelblue',
        linewidth=1.2, alpha=0.7, label='30-min (original)')
ax.plot(df_60min['datetime'], df_60min[demand_col], color='tomato',
        linewidth=2.0, linestyle='--', label='60-min (downsampled)')
ax.set_xlabel('Date')
ax.set_ylabel('Demand (MW)')
ax.set_title('DataLoader — 30-min vs 60-min resampling (first week of Jan 2024)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"30-min rows: {len(df_30min)},  60-min rows: {len(df_60min)}")

---

## Summary

| API | Purpose |
|---|---|
| `DataLoader()` | Load parquet datasets from `powerzoo/data/parquet/` |
| `loader.load_data(dataset_name, start_date, end_date)` | Date-filtered load with optional resampling |
| `loader.load_data(columns=[...])` | Auto-detect dataset from column names |
| `loader.load_data(resample='60min')` | Downsample / upsample to target frequency |
| `class MyReward(RewardFunction)` | Custom reward: implement `compute(state, info)` |
| `register_task(name, MyTask, force=True)` | Add task to global registry |
| `make_task_env('my_task')` | Use after `register_task` |

**Available bundled datasets** (identifiers in `powerzoo/data/`):
- `GB_NESO_Demand_2009_2025_30min` — system demand (2009–2025)
- `GB_Gen_by_Type_2016_2025_30min` — generation by type: Wind, Solar, Nuclear, Gas, …
- `GB_Forecast_Actual_Demand_2023_2025_30min` — day-ahead forecast vs actual demand

## You've completed the core tutorial series!

The recommended learning path:

```
NB01 (Quickstart)
  → NB02 (SB3 plugin)
    → NB03 (Anatomy)
      → NB04/05/06 (Three tasks)
        → NB07 (Evaluation)
          → NB08 (time-series & custom tasks — this notebook)
```

For questions or contributions: [GitHub Issues](https://github.com/powerzoojax/PowerZooPy/issues)